### Proceso de vectorización basado en subsecuencias (*k*-mers) de PETases

Este proceso consta de varios pasos:
- Leectura de las secuencias proteicas
- Conversión de las secuencias en subsecuencias de tamaño fijo (*k*-mers)
- Transformación los *k*-mers en vectores númerico mendiante TF-IDF
- Construcción de una matriz de características (*feature matrix*)
- Exportar la matriz para su uso en modelos predictivos

In [1]:
import numpy as np
import pandas as pd
from pandas import read_excel
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Cargamos la tabla de PETasas
tabla = pd.read_excel("Tabla_PETases.xlsx", sheet_name="PETases")
tabla.head()

,qseqid,sseqid,pident,qlen,slen,length,qstart,qend,sstart,send,...,Isolation Source,true_extreme_environment,number_extr_environments,type_env_extr,environment,env comments,Final especie,true_micr_extreme,microorg_extremophile,microorg cooments
0,GCA_913061175.1_SRR3933257_bin.1_MetaBAT_v2.12...,P0173,71.111,314,319,315,1,313,4,318,...,marine metagenome,no,NaN,NaN,NaN,NaN,uncultured Psychrobacter sp.,NaN,NaN,NaN
1,AIK49165.1,P0111,81.768,214,205,181,34,214,25,205,...,Rhizosphere,no,NaN,NaN,NaN,NaN,Bacillus atrophaeus subsp. globigii,NaN,NaN,NaN
2,GCA_002345575.1_ASM234557v1_genomic_00658,P0176,73.733,218,218,217,1,217,1,217,...,metal/plastic,no,NaN,NaN,NaN,NaN,Stutzerimonas stutzeri,NaN,NaN,NaN
3,GCA_002345575.1_ASM234557v1_genomic_03654,P0174,67.662,201,201,201,1,201,1,201,...,metal/plastic,no,NaN,NaN,NaN,NaN,Stutzerimonas stutzeri,NaN,NaN,NaN
4,MBX7275218.1,P0176,84.332,218,218,217,1,217,1,217,...,ice melting water from glacier site,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,NaN,NaN,NaN


In [ ]:
# Seleccionamos las PETases que son extremas y extremotelerantes
PETases_extreme = tabla[(tabla["true_extreme_environment"] == "extreme")]
PETases_extreme.head()

,qseqid,sseqid,pident,qlen,slen,length,qstart,qend,sstart,send,...,Isolation Source,true_extreme_environment,number_extr_environments,type_env_extr,environment,env comments,Final especie,true_micr_extreme,microorg_extremophile,microorg cooments
4,MBX7275218.1,P0176,84.332,218,218,217,1,217,1,217,...,ice melting water from glacier site,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,NaN,NaN,NaN
5,MBX7275838.1,P0174,72.139,201,201,201,1,201,1,201,...,ice melting water from glacier site,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,NaN,NaN,NaN
6,GCA_021819735.1_ASM2181973v1_genomic_01925,P0159,72.587,280,281,259,14,266,2,259,...,acid mine drainage sediment,extreme,monoextremophile,acidophile,acid mine drainage,NaN,Deltaproteobacteria bacterium,NaN,NaN,NaN
7,GCA_021819735.1_ASM2181973v1_genomic_01926,P0159,75.290,279,281,259,14,265,2,259,...,acid mine drainage sediment,extreme,monoextremophile,acidophile,acid mine drainage,NaN,Deltaproteobacteria bacterium,NaN,NaN,NaN
8,GCA_029937415.1_ASM2993741v1_genomic_01175,P0137,78.599,283,264,257,27,283,4,260,...,hydrothermal plume,extreme,polyextremophile,"thermophile, piezophile",hydrothermal vent,NaN,Longimicrobiales bacterium,NaN,NaN,NaN


In [ ]:
# Seleccionamos las PETases thermofilas (también las poliextremofilias)
PETases_thermo = PETases_extreme[(PETases_extreme["type_env_extr"] == "thermophile") | (PETases_extreme["type_env_extr"] == "thermophile, piezophile") 
                                 | (PETases_extreme["type_env_extr"] == "thermophile, alkalophile")]

PETases_thermo.head()



,qseqid,sseqid,pident,qlen,slen,length,qstart,qend,sstart,send,...,Isolation Source,true_extreme_environment,number_extr_environments,type_env_extr,environment,env comments,Final especie,true_micr_extreme,microorg_extremophile,microorg cooments
8,GCA_029937415.1_ASM2993741v1_genomic_01175,P0137,78.599,283,264,257,27,283,4,260,...,hydrothermal plume,extreme,polyextremophile,"thermophile, piezophile",hydrothermal vent,NaN,Longimicrobiales bacterium,NaN,NaN,NaN
14,PDM41723.1,P0258,68.016,247,249,247,2,247,1,247,...,hot spring,extreme,monoextremophile,thermophile,hot spring,NaN,Parageobacillus yumthangensis,NaN,NaN,NaN
15,MCX7662332.1,P0279,66.011,355,352,356,1,355,1,352,...,alkaline hot spring water,extreme,polyextremophile,"thermophile, alkalophile",hot spring,NaN,Tepidimonas fonticaldi,NaN,NaN,NaN
43,GCA_002332825.1_ASM233282v1_genomic_00638,P0279,60.784,356,352,357,1,356,1,352,...,Von Damm hydrothermal vent plume at depth 2238m,extreme,polyextremophile,"thermophile, piezophile",hydrothermal vent,NaN,Cupriavidus sp. UBA2027,NaN,NaN,NaN
56,AKS39435.1,P0258,65.992,245,249,247,1,245,1,247,...,hot spring,extreme,monoextremophile,thermophile,hot spring,NaN,Anoxybacillus gonensis,NaN,NaN,NaN


In [ ]:
# Pasamos la PETasas termófilas a un excel 
PETases_thermo.to_excel("PETases_thermophiles.xlsx", index=0)

### 1. Convertimos las secuencias en *k*-mers

In [ ]:
# Definimos la función de los k-mers
def kmer(seq,seq_length):
    kmer_words = [seq[i:i+seq_length] for i in range(len(seq)-seq_length+1)]
    return ' '.join(kmer_words)

In [ ]:
# Definimos longitud del k-mer
k = 3
# Convertimos las seq en kmeros y creamos una columna al mismo tiempo en el df
PETases_thermo["kmer-seq"] = PETases_thermo["query_seq"].apply(lambda seq: kmer(seq, k))

### 2. Feature Extraction

En este paso, las secuencias proteicas se transforman en representaciones numéricas mediante un enfoque estadístico basado en *k*-mers y TF-IDF (*Term Frequency–Inverse Document Frequency*). A diferencia de los modelos de lenguaje de proteínas profundos (*protein Language Models*, pLMs), que aprenden representaciones automáticamente, este enfoque clásico permite interpretar directamente qué subsecuencias de aminoácidos contribuyen más a la clasificación.


La metodología permite:

- Identificar *k*-mers potencialmente asociados a PETasas termófilas
- Construir matrices de características para modelos de clasificación supervisada
- Comparar patrones de composición entre proteínas termófilas y no termófilas

La vectorización TF-IDF asigna un peso a cada *k*-mer según:
- **TF (Term Frequency):** frecuencia de aparición del *k*-mer en una proteína concreta
- **IDF (Inverse Document Frequency):** especificidad del *k*-mer dentro del conjunto total de proteínas

De este modo, los *k*-mers frecuentes en una proteína pero poco comunes en el resto reciben mayor peso informativo.

In [ ]:
vectorizador = TfidfVectorizer()

# Creamos una matriz dispersa (mayoría de valores son 0)
matriz_tfidf = vectorizador.fit_transform(PETases_thermo["kmer-seq"])

# Obtenemos los k-mers
feat_names = vectorizador.get_feature_names_out()

# Ids de las proteinas
PETases_thermo = PETases_thermo.set_index("qseqid")

# Transformamos en una tabla legible con la matriz dispersa y los diferentes kmers
DFFE = pd.DataFrame(data=matriz_tfidf.toarray(), index=PETases_thermo.index, columns=feat_names)

DFFE.to_csv("PETases_kmers_07ABR26.csv", sep="\t")


El resultado final es una matriz dispersa (*sparse matrix*) donde:
- Las filas representan proteínas
- Las columnas representan *k*-mers detectados
- Los valores corresponden a los pesos TF-IDF calculados

Dado el elevado número posible de combinaciones de aminoácidos (20³ = 8000 posibles *3-mers*), la mayoría de posiciones contienen valores cero, lo cual es esperado en este tipo de representación.

### Construcción del conjunto de entrenamiento

Se repitió el mismo pipeline de vectorización para PETasas procedentes de organismos no termófilos con el objetivo de generar un conjunto de entrenamiento balanceado.

Finalmente, se obtuvo un dataset inicial compuesto por:
- 150 PETasas termófilas
- 150 PETasas no termófilas

Esta matriz de características se utilizó posteriormente para el entrenamiento del modelo XGBoost.

In [6]:
# Seleccionamos las PETases NO thermofilas 
PETases_nothermo = PETases_extreme[(PETases_extreme["type_env_extr"] != "thermophile") & (PETases_extreme["type_env_extr"] != "thermophile, piezophile") 
                                 & (PETases_extreme["type_env_extr"] != "thermophile, alkalophile")]

PETases_nothermo.head()


,qseqid,sseqid,pident,qlen,slen,length,qstart,qend,sstart,send,...,Isolation Source,true_extreme_environment,number_extr_environments,type_env_extr,environment,env comments,Final especie,true_micr_extreme,microorg_extremophile,microorg cooments
4,MBX7275218.1,P0176,84.332,218,218,217,1,217,1,217,...,ice melting water from glacier site,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,NaN,NaN,NaN
5,MBX7275838.1,P0174,72.139,201,201,201,1,201,1,201,...,ice melting water from glacier site,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,NaN,NaN,NaN
6,GCA_021819735.1_ASM2181973v1_genomic_01925,P0159,72.587,280,281,259,14,266,2,259,...,acid mine drainage sediment,extreme,monoextremophile,acidophile,acid mine drainage,NaN,Deltaproteobacteria bacterium,NaN,NaN,NaN
7,GCA_021819735.1_ASM2181973v1_genomic_01926,P0159,75.290,279,281,259,14,265,2,259,...,acid mine drainage sediment,extreme,monoextremophile,acidophile,acid mine drainage,NaN,Deltaproteobacteria bacterium,NaN,NaN,NaN
16,MCG8397989.1,P0111,82.873,214,205,181,34,214,25,205,...,hypersaline environment,extreme,monoextremophile,halophile,hypersaline environment (soil),NaN,Bacillus atrophaeus,NaN,NaN,NaN


In [7]:
# Nos quedamos con solo con las primeras 200 PETases
PETases_nothermo = PETases_nothermo[:201]
PETases_nothermo

,qseqid,sseqid,pident,qlen,slen,length,qstart,qend,sstart,send,...,Isolation Source,true_extreme_environment,number_extr_environments,type_env_extr,environment,env comments,Final especie,true_micr_extreme,microorg_extremophile,microorg cooments
4,MBX7275218.1,P0176,84.332,218,218,217,1,217,1,217,...,ice melting water from glacier site,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,NaN,NaN,NaN
5,MBX7275838.1,P0174,72.139,201,201,201,1,201,1,201,...,ice melting water from glacier site,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,NaN,NaN,NaN
6,GCA_021819735.1_ASM2181973v1_genomic_01925,P0159,72.587,280,281,259,14,266,2,259,...,acid mine drainage sediment,extreme,monoextremophile,acidophile,acid mine drainage,NaN,Deltaproteobacteria bacterium,NaN,NaN,NaN
7,GCA_021819735.1_ASM2181973v1_genomic_01926,P0159,75.290,279,281,259,14,265,2,259,...,acid mine drainage sediment,extreme,monoextremophile,acidophile,acid mine drainage,NaN,Deltaproteobacteria bacterium,NaN,NaN,NaN
16,MCG8397989.1,P0111,82.873,214,205,181,34,214,25,205,...,hypersaline environment,extreme,monoextremophile,halophile,hypersaline environment (soil),NaN,Bacillus atrophaeus,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
907,TRL78704.1,P4615,60.969,749,766,784,1,746,1,761,...,desert soil,extreme,monoextremophile,xerophile,desert,NaN,Staphylococcus haemolyticus,NaN,NaN,NaN
913,GCA_000423205.1_ASM42320v1_genomic_00898,P0246,68.794,283,284,282,1,282,2,283,...,Soda lake mud,extreme,monoextremophile,alkalophile,soda lake,NaN,Jonesia quinghaiensis DSM 15701,NaN,NaN,NaN
914,GCA_000423205.1_ASM42320v1_genomic_02424,P0246,77.385,286,284,283,3,285,1,283,...,Soda lake mud,extreme,monoextremophile,alkalophile,soda lake,NaN,Jonesia quinghaiensis DSM 15701,NaN,NaN,NaN
918,TBN35693.1,P0176,90.826,218,218,218,1,218,1,218,...,Glacier ice,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. BGI-2,NaN,NaN,NaN


In [ ]:
# Unificamos las PETasas
# Primero añadimos la columna "observations" a cada df
PETases_thermo["Observations"] = "termophile"
PETases_nothermo["Observations"] = "no thermophile"

PETases_nothermo.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_7480\2904479.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PETases_thermo["Observations"] = "termophile"


,qseqid,sseqid,pident,qlen,slen,length,qstart,qend,sstart,send,...,true_extreme_environment,number_extr_environments,type_env_extr,environment,env comments,Final especie,true_micr_extreme,microorg_extremophile,microorg cooments,Observations
4,MBX7275218.1,P0176,84.332,218,218,217,1,217,1,217,...,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,NaN,NaN,NaN,no thermophile
5,MBX7275838.1,P0174,72.139,201,201,201,1,201,1,201,...,extreme,monoextremophile,psychrophile,cryosphere,NaN,Pseudomonas sp. ERGC3:01,NaN,NaN,NaN,no thermophile
6,GCA_021819735.1_ASM2181973v1_genomic_01925,P0159,72.587,280,281,259,14,266,2,259,...,extreme,monoextremophile,acidophile,acid mine drainage,NaN,Deltaproteobacteria bacterium,NaN,NaN,NaN,no thermophile
7,GCA_021819735.1_ASM2181973v1_genomic_01926,P0159,75.290,279,281,259,14,265,2,259,...,extreme,monoextremophile,acidophile,acid mine drainage,NaN,Deltaproteobacteria bacterium,NaN,NaN,NaN,no thermophile
16,MCG8397989.1,P0111,82.873,214,205,181,34,214,25,205,...,extreme,monoextremophile,halophile,hypersaline environment (soil),NaN,Bacillus atrophaeus,NaN,NaN,NaN,no thermophile


In [ ]:
# Concatenamos las tablas una debajo de otra
PETases_combo = pd.concat([PETases_thermo, PETases_nothermo])

In [ ]:
# Definimos longitud del k-mer
k = 3

# Calculamos los k-mers
PETases_combo["kmer-seq"] = PETases_combo["query_seq"].apply(lambda seq: kmer(seq, k))

PETases_combo.head()

,qseqid,sseqid,pident,qlen,slen,length,qstart,qend,sstart,send,...,number_extr_environments,type_env_extr,environment,env comments,Final especie,true_micr_extreme,microorg_extremophile,microorg cooments,Observations,kmer-seq
8,GCA_029937415.1_ASM2993741v1_genomic_01175,P0137,78.599,283,264,257,27,283,4,260,...,polyextremophile,"thermophile, piezophile",hydrothermal vent,NaN,Longimicrobiales bacterium,NaN,NaN,NaN,termophile,MNR NRW RWF WFF FFP FPR PRV RVL VLL LLL LLA LA...
14,PDM41723.1,P0258,68.016,247,249,247,2,247,1,247,...,monoextremophile,thermophile,hot spring,NaN,Parageobacillus yumthangensis,NaN,NaN,NaN,termophile,MMK MKI KIV IVP VPP PPK PKP KPF PFF FFF FFE FE...
15,MCX7662332.1,P0279,66.011,355,352,356,1,355,1,352,...,polyextremophile,"thermophile, alkalophile",hot spring,NaN,Tepidimonas fonticaldi,NaN,NaN,NaN,termophile,MTH THQ HQL QLT LTI TIE IEP EPL PLG LGA GAA AA...
43,GCA_002332825.1_ASM233282v1_genomic_00638,P0279,60.784,356,352,357,1,356,1,352,...,polyextremophile,"thermophile, piezophile",hydrothermal vent,NaN,Cupriavidus sp. UBA2027,NaN,NaN,NaN,termophile,MTY TYQ YQL QLT LTI TIE IEP EPL PLG LGQ GQT QT...
56,AKS39435.1,P0258,65.992,245,249,247,1,245,1,247,...,monoextremophile,thermophile,hot spring,NaN,Anoxybacillus gonensis,NaN,NaN,NaN,termophile,MKL KLV LVA VAP APQ PQP QPF PFT FTF TFE FEA EA...


In [ ]:
vectorizador = TfidfVectorizer()

# Creamos una matriz dispersa (mayoría de valores son 0)
matriz_tfidf_combo = vectorizador.fit_transform(PETases_combo["kmer-seq"])

# Obtenemos los kmers
feat_names_combo = vectorizador.get_feature_names_out()

# Ids de las proteinas
PETases_combo = PETases_combo.set_index("qseqid")

# Transformamos en una tabla legible con la matriz dispersa y los diferentes kmers
DFFE_combo = pd.DataFrame(data=matriz_tfidf_combo.toarray(), index=PETases_combo.index, columns=feat_names_combo)
DFFE_combo["Observations"] = PETases_combo["Observations"]

DFFE_combo.to_excel("PETases_kmers_combo_07ABR26.xlsx")

### Vectorización de las PETases de Uniprot

Se utilizó un nuevo conjunto de secuencias para evaluar la capacidad predictiva del modelo entrenado sobre datos externos. Las secuencias se obtuvieron desde UniProtKB utilizando el identificador enzimático *EC 3.1.1.101*, incluyendo tanto entradas revisadas como no revisadas, obteniendo un total de 473 PETasas.

Posteriormente, todas las secuencias fueron transformadas en *k*-mers y vectorizadas siguiendo el mismo pipeline aplicado previamente al conjunto de entrenamiento.

In [3]:
uniprot_PET = pd.read_excel("C:/Users/HP/OneDrive/Escritorio/Practicas_pLMs/uniprotkb_PETases.xlsx")
uniprot_PET.head()

,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Sequence
0,A0A0K8P6T7,reviewed,PETH_PISS1,Poly(ethylene terephthalate) hydrolase (PET hy...,ISF6_4831,Piscinibacter sakaiensis (Ideonella sakaiensis),290,MNFPRASRLMQAAVLGGLMAVSAAATAQTNPYARGPNPTAASLEAS...
1,E9LVH9,reviewed,PETH2_THECS,Cutinase 2 (EC 3.1.1.74) (Poly(ethylene tereph...,cut2,Thermobifida cellulosilytica,262,MANPYERGPNPTDALLEARSGPFSVSEERASRFGADGFGGGTIYYP...
2,F7IX06,reviewed,PETH2_THEAE,Cutinase est2 (EC 3.1.1.74) (Poly(ethylene ter...,est2 est119,Thermobifida alba (Thermomonospora alba),300,MSVTTPRRETSLLSRALRATAAAATAVVATVALAAPAQAANPYERG...
3,G8GER6,reviewed,PETH1_THEFU,Cutinase cut1 (EC 3.1.1.74) (Poly(ethylene ter...,cut_1,Thermobifida fusca (Thermomonospora fusca),319,MPPHAARPGPAQNRRGRAMAVITPRRERSSLLSRALRFTAAAATAL...
4,G9BY57,reviewed,PETH_UNKP,Leaf-branch compost cutinase (LC-cutinase) (LC...,NaN,Unknown prokaryotic organism,293,MDGVLWRVRTAALMAALLALAAWALVWASPSVEAQSNPYQRGPNPT...


In [ ]:
# Definimos longitud del k-mer
k = 3
# Convertimos las seq en kmeros y creamos una columna al mismo tiempo en el df
uniprot_PET["kmer-seq"] = uniprot_PET["Sequence"].apply(lambda seq: kmer(seq, k))

In [ ]:
vectorizador = TfidfVectorizer()

# Creamos una matriz dispersa (mayoría de valores son 0)
matriz_tfidf_uniprot = vectorizador.fit_transform(uniprot_PET["kmer-seq"])

# Obtenemos los k-mers
feat_names_uniprot = vectorizador.get_feature_names_out()

# Ids de las proteinas
uniprot_PET = uniprot_PET.set_index("Entry Name")

# Transformamos en una tabla legible con la matriz dispersa y los diferentes kmers
DFFE_uniprot = pd.DataFrame(data=matriz_tfidf_uniprot.toarray(), index=uniprot_PET, columns=feat_names_uniprot)

DFFE_uniprot.to_excel("PETases_kmers_uniprot_17ABR26.xlsx")